<a href="https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/blob/main/Day3_Evasion_Lab_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 3 — Evasion Lab
## Walking Past the Detector You Built Yesterday

**What this lab is about.** Yesterday, in Lab 3c, you trained a **Decision Tree** — a flow-chart of
yes/no questions that the computer wrote for itself — to look at a login record and answer one
question: *"did this come from an attacker?"* It caught 158 of the 177 attacks in the exam set.

Today you become the attacker. You are going to change **one** thing about your attack — not how you
attack, just where you appear to be coming from — and see whether the Decision Tree still notices.

**You will never touch the Decision Tree.** It is not retrained and not modified. That is what makes
this an *evasion* attack: the attacker changes the **input**, never the model.

**How to run this:** click into the first cell and press `Shift + Enter`. Keep pressing it. Read the
output of each cell before moving on. Nothing here installs anything, and nothing here touches a real
system.

---

## Part 1 — Rebuild yesterday's Decision Tree

Colab forgets everything between sessions, so we rebuild the exact model from Lab 3c. Same data,
same five features, same random seed — so your numbers will match yesterday's to the decimal.

In [ ]:
# ---- The toolboxes we need -------------------------------------------------------
# pandas      : works with tables of data (a table is called a "DataFrame")
# numpy       : fast maths on lots of numbers at once
# matplotlib  : draws charts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- The machine-learning pieces, all from scikit-learn --------------------------
# scikit-learn ("sklearn") is the standard Python toolkit for classic machine learning.
#   train_test_split      : splits rows into a "learn from these" pile and an "exam" pile
#   MinMaxScaler          : squeezes every number onto the same 0-1 ruler
#   DecisionTreeClassifier: the flow-chart model itself
#   the metrics           : the scoring functions from Day 2 section 2.7
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import MinMaxScaler
from sklearn.tree            import DecisionTreeClassifier
from sklearn.metrics         import precision_score, recall_score, confusion_matrix

print("Toolboxes loaded. Nothing to install - Colab already has all of these.")

In [ ]:
# ---- Where the data comes from ---------------------------------------------------
# This is the SAME login log you used in Lab 3c. Nothing new to download by hand.
# Note the address is "raw.githubusercontent.com". A normal github.com link returns a
# web PAGE, which pandas cannot read.
DATA_URL  = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/day2_auth_logs.csv"
DATA_FILE = "day2_auth_logs.csv"     # only used by the offline fallbacks below

import os   # lets us ask the computer whether a file exists on disk

# ---- Same loader as Labs 3a, 3b and 3c: internet first, then local file, then upload box ----
def load_csv(url="", fname=""):
    """Load the CSV from a URL, or a local file, or (on Colab) an upload box."""
    # parse_dates= tells pandas that this column holds a date and time, not text,
    # so we can later ask it "what hour of the day was this?"
    if url:
        try:
            return pd.read_csv(url, parse_dates=["Login Timestamp"])
        except Exception as problem:
            print("Could not read from the internet:", problem)
            print("Falling back to a local copy...")
    for path in [fname, os.path.join("data", fname)]:
        if fname and os.path.exists(path):
            return pd.read_csv(path, parse_dates=["Login Timestamp"])
    # Last resort, Colab only: pop up a file picker.
    from google.colab import files
    up = files.upload()
    return pd.read_csv(list(up.keys())[0], parse_dates=["Login Timestamp"])

logs = load_csv(url=DATA_URL, fname=DATA_FILE)

print("Loaded {:,} login attempts.".format(len(logs)))
print("Of those, {:,} came from IP addresses known to belong to attackers."
      .format(int(logs["Is Attack IP"].sum())))

In [ ]:
# ---- The LABEL: the thing we are asking the model to predict ---------------------
# .astype(int) turns True/False into 1/0, because the model wants numbers.
label = logs["Is Attack IP"].astype(int)

# ---- The FIVE features, exactly as in Lab 3c -------------------------------------
# A "feature" is one piece of information about a login that the model is allowed to use.
ua_text = logs["User Agent String"].fillna("").str.lower()   # blank-safe, lower-case

features = pd.DataFrame({
    # 1. Did the login work? 1 = yes, 0 = no.
    "login_successful": logs["Login Successful"].astype(int),

    # 2. Does the user-agent name a scripting tool rather than a real browser?
    #    The | character means OR, so this asks "does it contain any of these words?"
    "is_scripted_ua": ua_text.str.contains(
        "python-requests|curl|go-http|wget|scrapy|okhttp").astype(int),

    # 3. Is the country anything other than Norway, this app's home country?
    "is_foreign": (logs["Country"] != "NO").astype(int),

    # 4. What hour of the day was it? .dt.hour pulls the hour out of the timestamp.
    "hour": logs["Login Timestamp"].dt.hour,
})

print("Four features built so far. The fifth one needs the train/test split first.")
print(features.head())

In [ ]:
# ---- Split the rows: some to learn from, some held back as an exam ---------------
# test_size=0.25  -> a quarter of the rows are held back
# random_state=42 -> fixes WHICH rows, so everyone in the room gets identical numbers
# stratify=label  -> keeps the attack/benign mix the same in both halves
train_rows, test_rows = train_test_split(
    logs.index, test_size=0.25, random_state=42, stratify=label)

# ---- The fifth feature: how many times has this IP failed before? ----------------
# IMPORTANT: we count this using TRAINING rows only. If we counted using the exam rows
# too, the model would be peeking at the answers - that mistake is called "leakage".
train_logs   = logs.loc[train_rows]
fails_per_ip = (train_logs.loc[train_logs["Login Successful"] == False]
                .groupby("IP Address").size())

# .map() looks each row's IP up in that table. An IP we never saw in training gets 0.
features["ip_fail_count"] = logs["IP Address"].map(fails_per_ip).fillna(0).astype(int)

X_train, X_test = features.loc[train_rows], features.loc[test_rows]
y_train, y_test = label.loc[train_rows],    label.loc[test_rows]

print("Learning from {:,} logins, holding back {:,} for the exam."
      .format(len(X_train), len(X_test)))
print("Attacks hidden in the exam pile: {:,}".format(int(y_test.sum())))

In [ ]:
# ---- Put every number on the same 0-1 ruler --------------------------------------
# The ruler is learned from the TRAINING rows only (the Lab 3b golden rule),
# then applied to both halves.
scaler  = MinMaxScaler().fit(X_train)
Xtr_s   = scaler.transform(X_train)
Xte_s   = scaler.transform(X_test)

# ---- Train the Decision Tree -----------------------------------------------------
# max_depth=4          -> at most four questions deep, so we can still read it
# class_weight="balanced" -> attacks are rare, so tell the tree to take them seriously
# random_state=42      -> same tree for everyone, every time
tree = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced")
tree.fit(Xtr_s, y_train)

# ---- Score it on the exam pile it has never seen ---------------------------------
baseline_pred = tree.predict(Xte_s)
tn, fp, fn, tp = confusion_matrix(y_test, baseline_pred, labels=[0, 1]).ravel()

print("YESTERDAY'S MODEL, REBUILT")
print("  attacks caught       : {:>4}".format(tp))
print("  attacks missed       : {:>4}".format(fn))
print("  false alarms         : {:>4}   <- innocent logins wrongly flagged".format(fp))
print("  RECALL    (of all real attacks, how many did we catch?) : {:.2f}".format(tp/(tp+fn)))
print("  PRECISION (of everything we flagged, how much was real?): {:.2f}".format(tp/(tp+fp)))
print("\nThis should match your Lab 3c numbers exactly.")

In [ ]:
# ---- Before attacking it, look at what it actually learned -----------------------
# A Decision Tree can tell us how much each of the five questions mattered.
# The numbers are shares and add up to 1.0.
importance = (pd.Series(tree.feature_importances_, index=X_train.columns)
              .sort_values(ascending=False))

print("What the Decision Tree leaned on:\n")
for name, value in importance.items():
    bar = "#" * int(round(value * 50))
    print("  {:<18} {:>6.1%}  {}".format(name, value, bar))

print("\nOne question is doing almost all of the work.")
print("Is that a mistake? Check the data before you decide:\n")
attacks = logs[logs["Is Attack IP"] == True]
benign  = logs[logs["Is Attack IP"] == False]
print("  attack logins that are foreign : {:.1%}".format((attacks["Country"] != "NO").mean()))
print("  ordinary logins that are foreign: {:.1%}".format((benign["Country"] != "NO").mean()))
print("\nThe pattern is real. The tree learned exactly what it was shown.")

---
## Part 2 — The attack: change exactly one thing

You are now the attacker. Same campaign, same tooling, same target accounts, same timing.

The only change: you rent a **Norwegian residential proxy** for a few dollars, so your logins now
appear to arrive from inside Norway instead of from abroad.

We simulate that in the data by setting `is_foreign` to 0 on those attack records — in plain English,
*"suppose these exact same logins had arrived from a Norwegian address."* Nothing else about them
changes, because in real life changing your exit IP would not change anything else either.

**The Decision Tree is not retrained and not touched.**

In [ ]:
# ---- Start from the attacks the Decision Tree CORRECTLY catches ------------------
# There is no point evading a detector on attacks it was already missing.
caught = y_test.index[(baseline_pred == 1) & (y_test == 1)]
X_caught = X_test.loc[caught]
print("Attacks the Decision Tree currently catches: {}".format(len(X_caught)))
print("These are the ones we now try to sneak past it.\n")

In [ ]:
# ---- Five things an attacker could change, one at a time -------------------------
# Each entry says which feature to overwrite and what to set it to.
mutations = {
    "1. Route through a Norwegian IP":              {"is_foreign": 0},
    "2. Use a real browser user-agent":             {"is_scripted_ua": 0},
    "3. Attack during business hours":              {"hour": 14},
    "4. Slow down / rotate IP addresses":           {"ip_fail_count": 1},
    "5. Succeed on the first attempt":              {"login_successful": 1},
}

rows = []
for name, change in mutations.items():
    mutated = X_caught.copy()               # take a fresh copy of the caught attacks
    for column, value in change.items():
        mutated[column] = value             # apply the one change

    # Ask the UNCHANGED Decision Tree to judge the mutated logins
    verdict = tree.predict(scaler.transform(mutated))
    evaded  = int((verdict == 0).sum())     # 0 means "the tree thinks this is fine"

    rows.append({"the one thing you changed": name,
                 "still caught": int((verdict == 1).sum()),
                 "evaded":       evaded,
                 "evasion rate": "{:.0%}".format(evaded / len(mutated))})

results = pd.DataFrame(rows)
print(results.to_string(index=False))
print("\nThe model was never retrained. Only the input changed.")

In [ ]:
# ---- The same thing as a picture -------------------------------------------------
rates  = [int(r["evaded"]) / len(X_caught) * 100 for r in rows]
names  = [r["the one thing you changed"] for r in rows]
colours = ["#C0392B" if v > 50 else "#5A6B85" for v in rates]

plt.figure(figsize=(9, 3.6))
plt.barh(names[::-1], rates[::-1], color=colours[::-1])
plt.xlabel("% of caught attacks that now slip past")
plt.xlim(0, 105)
plt.title("One change defeats it. The other four do almost nothing.")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 3 — "So just retrain it?"

That is the question everyone asks next, and it is a fair one. So let's do it.

We assume the attacker has permanently moved to Norwegian IPs, and we rebuild the Decision Tree
from scratch on that new reality. Does it recover?

In [ ]:
# ---- Scenario: every attack now arrives from a Norwegian IP ----------------------
Xtr_moved = X_train.copy();  Xtr_moved.loc[y_train == 1, "is_foreign"] = 0
Xte_moved = X_test.copy();   Xte_moved.loc[y_test  == 1, "is_foreign"] = 0

def score(model, scl, X, y, tag):
    """Score any model on any set of rows and return one tidy row of results."""
    tn, fp, fn, tp = confusion_matrix(y, model.predict(scl.transform(X)), labels=[0, 1]).ravel()
    return {"scenario": tag,
            "recall":    round(tp / (tp + fn), 2),
            "caught":    tp, "missed": fn, "false alarms": fp}

# a) the original tree, before the attacker moved
before = score(tree, scaler, X_test, y_test, "Day-2 tree, as you trained it")

# b) the SAME tree, now that the attacker has moved - no retraining
during = score(tree, scaler, Xte_moved, y_test, "Attacker moves to a Norwegian IP")

# c) a brand-new tree, retrained from scratch on the new reality
scaler2 = MinMaxScaler().fit(Xtr_moved)
tree2   = DecisionTreeClassifier(max_depth=4, random_state=42,
                                 class_weight="balanced").fit(scaler2.transform(Xtr_moved), y_train)
after   = score(tree2, scaler2, Xte_moved, y_test, "Retrained on the new reality")

print(pd.DataFrame([before, during, after]).to_string(index=False))

print("\nWhat the retrained tree leans on now:")
for name, value in (pd.Series(tree2.feature_importances_, index=Xtr_moved.columns)
                    .sort_values(ascending=False)).items():
    print("  {:<18} {:>6.1%}".format(name, value))

---
## What you just learned

**1. The 0.89 was never real detection.** It was borrowed from a correlation the attacker controls.
Strip geography away and the honest capability of these five columns is about 0.21. That is a
**data** problem, not a model problem — retraining cannot fix it.

**2. But the retrained tree is far cheaper to live with.** False alarms fall from 669 to 32. The old
one wasted roughly four alerts for every real catch; this one wastes under one. It catches less, and
is worth far more of an analyst's day.

**3. The fix is better features, not more retraining.** With geography gone, the tree fell back on
`ip_fail_count` — failure bursts per IP, a genuine *behavioural* signal and exactly the right
direction. There simply is not enough of that kind of signal in five columns.

> **The one line to remember:** you cannot retrain your way out of a feature problem. Pick signals the
> attacker cannot buy their way past — and read feature importance **before** you ship, not after
> you are breached.